<a href="https://colab.research.google.com/github/ekairaliev/AppStat-Lab01/blob/main/AppStat_Lab01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">

<p>Университет ИТМО</p>

<br>

<p align="center">
  <font size="5"><b>Лабораторная работа № 1</b></font>
</p>

<p>по прикладной статистике</p>

<br>

<h3>
Статистический анализ распределений температуры воспламенения,
задержки воспламенения и скорости горения металлических материалов
</h3>

</div>

<br>

<div align="right">

<p>
Выполнили:<br>
Кайралиев Эмиль Салаватович, группа О3243<br>
Торхова Кристина Сергеевна, группа P3269
</p>

<p>
Преподаватель:<br>
Киселёв Константин Сергеевич
</p>

</div>

<br>

<div align="center">
<p>Санкт-Петербург<br>2026</p>
</div>

# Предисловие

Прежде чем перейти к самой лабораторной работе, мы хотели бы немного рассказать о том, как решили подойти к её выполнению.

Основой для нашей работы стала книга Аллена Дауни «Думай как аналитик» [1] *. Прочитать около 300 страниц за три дня оказалось не самой простой задачей, особенно учитывая, что многие понятия и методы статистики были для нас новыми. Но материал, преподнесенный в научпоп формате, очень увлёк.

Однако Дауни сознательно уделяет мало внимания математическим выкладкам, что мы считаем небольшим недостатком книги. Поэтому в качестве второго основного источника мы решили использовать учебное пособие М. Б. Лагутина «Наглядная математическая статистика» [2].

Ещё одна деталь, которая нам особенно понравилась в книге «Думай как аналитик», — глоссарий в конце каждой главы. Во время чтения оказалось очень удобно возвращаться к кратким определениям уже изученных понятий, особенно когда новые термины начинали пересекаться с предыдущими. Поэтому мы решили сделать нечто похожее и в нашей лабораторной работе. В конце будет отдельный раздел «Глоссарий», куда мы постепенно будем добавлять основные статистические термины, встречающиеся по ходу выполнения работы.

*Интересный факт: в книге одним из примеров датасетов является БД арктических пингвинов из HW1 :)

# Lab01 - sprint 1

##Random Variable 01 (температура воспламенения)

###Description

Для выполнения лабораторной работы мы довольно долго искали подходящий набор данных в журнале *Scientific Data*, который издаёт подразделение Nature Research (издательство Springer Nature). Частая проблема датасетов с хим. данными - это их сгенерированность (поясню: около 80% бд из журнала - результаты DFT/ML предсказаний химических свойств молекул, реакций), поэтому искали article с экспериментальными данными. После рассмотрения нескольких вариантов мы остановились на статье Penglin Wang из Пекинского технологического университета *An integrated database of combustion properties of metallic materials* (Интегрированная база данных о свойствах горения металлических материалов) [3],[4]. Характеристики горения зависят от множества факторов, поэтому нам показалось, что здесь будет интересно разобраться, какие закономерности можно обнаружить с помощью матстата.

В статье авторы собрали и систематизировали результаты исследований воспламенения и горения металлических материалов. Они рассмотрели более 160 научных источников и отобрали данные из 45 публикаций, объединив их в базу из 725 записей. В ней представлены чистые металлы и различные сплавы, в том числе на основе алюминия, магния, титана и железа. Авторы выделили пять основных характеристик: энтальпию горения, температуру воспламенения, время задержки воспламенения, скорость горения и пороговое давление кислорода. Помимо самих характеристик, в таблицах содержится информация о химическом составе материалов и условиях проведения экспериментов. Часть значений была взята непосредственно из опубликованных таблиц, а часть получена с помощью оцифровки графиков.

Небольшой ликбез для нехимиков: горение металла представляет собой экзотермическую окислительно-восстановительную реакцию, чаще всего с кислородом. Например, при горении магния протекает реакция:

2Mg + O₂ → 2MgO

В результате образуется оксид магния и выделяется теплота. При этом процесс не всегда начинается сразу. На поверхности металла может находиться оксидная плёнка, которая препятствует доступу кислорода и замедляет дальнейшее окисление. При нагревании скорость реакций увеличивается, оксидный слой может разрушаться, а выделение теплоты становится интенсивнее. Если теплота выделяется быстрее, чем отводится в окружающую среду, возможно резкое повышение температуры и переход к горению.

Первой случайной величиной мы выбрали **температуру воспламенения** (*ignition temperature*). В рассматриваемой статье под ней понимается минимальная экспериментально зарегистрированная температура, при которой наблюдается наиболее быстрое изменение температуры образца и интенсивности его свечения. Проще говоря, это температура, при которой в условиях конкретного эксперимента фиксируется начало интенсивного воспламенения.

Интересно, что температура воспламенения не является такой же однозначной характеристикой материала, как, например, температура плавления чистого металла при заданном давлении. Она зависит не только от химического состава, но и от состояния поверхности, геометрии образца, скорости нагревания, кислородной среды и метода испытания. Поэтому один и тот же материал при разных условиях может воспламеняться при разных температурах.

Но можно ли считать температуру воспламенения случайной величиной? Даже при одинаковых заданных условиях результаты повторных экспериментов могут немного различаться из-за неоднородности образцов, состояния оксидной плёнки, небольших колебаний условий и погрешностей измерений. При этом важно отличать такую случайную изменчивость от закономерных различий, вызванных, например, изменением состава сплава или давления кислорода.

Таким образом, у нас возникает несколько вопросов. Как распределены значения температуры воспламенения в собранной базе? Насколько сильно они различаются? Можно ли описать их одним распределением или сначала потребуется разделить наблюдения на группы? И какую часть изменчивости можно связать с условиями эксперимента?

Для начала посмотрим, как устроены исходные данные, и попробуем разобраться, с чем именно нам предстоит работать.

### EDA & Distribution

### Independence & Estimations

## Random Variable 02 (задержка воспламенения)

### Description

### EDA & Distribution

### Independence & Estimations

## Random Variable 03 (линейная скорость горения)

### Description

### EDA & Distribution

### Independence & Estimations

## Lab resume

### cat_mark_table

#Глоссарий

# Список использованных источников

1. Дауни А. Думай как аналитик. Статистика и данные с примерами на Python. 3-е изд. / пер. с англ. Е. Суворкин; науч. ред. К. Быков. — Астана : Спринт Бук, 2026. — 304 с. : ил. — ISBN 978-601-12-8917-7.

2. Лагутин М. Б. Наглядная математическая статистика : учебное пособие. — 2-е изд., испр. — М. : БИНОМ. Лаборатория знаний, 2009. — 472 с. : ил. — ISBN 978-5-94774-996-0.

3. Wang P., Ke H., Xue Y. An integrated database of combustion properties of metallic materials // Scientific Data. — 2026. — Vol. 13. — Article 460. — DOI: 10.1038/s41597-026-06862-8.

4. Wang P., Ke H., Xue Y. CombustionData [Электронный ресурс]. — GitHub, 2026. — URL: https://github.com/wpl2000/CombustionData.